# 🐑 Sheep Activity Classifier v4

### Cambios respecto a v3
- **417 videos** de train (antes 100) → permite backbone más potente
- **EfficientNet-B2** (9.1M params, input 260px) en lugar de B0 — mejor capacidad sin overfitting con 417 videos
- **`n_frames` = 12** (antes 8) — más contexto temporal con más datos disponibles
- **YOLO padding 40%** (antes 15%) — preserva contexto espacial clave para distinguir Sitting/Standing/Grazing
- **Pseudo-labeling con umbral adaptativo** — si confianza media < 0.6, baja automáticamente el umbral
- **Fine-tuning progresivo en 3 fases** en lugar de 2
- **Entrenamiento final sobre todos los datos** guardando mejor época
- Rutas de Kaggle preservadas exactamente


In [ ]:
!apt-get update && apt-get install -y aria2

In [ ]:
!aria2c -x 16 -s 16 -j 4 "https://data.mendeley.com/public-api/zip/h5ppwx6fn4/download/1" -o dataset.zip

In [ ]:
!unzip 'dataset.zip' -d ovejitas_dataset_2

In [ ]:
rm dataset.zip

In [ ]:
!unzip '/kaggle/working/ovejitas_dataset_2/Video Dataset of Sheep Activity (Grazing, Running, Sitting)/Grazing_Running_Sitting Sheep_Classes.zip' -d dataset_2

In [ ]:
rm -rf ovejitas_dataset_2

In [ ]:
import os
import shutil
import pandas as pd
from pathlib import Path

# 1. Configuración de rutas
# Ajusta SOURCE_DIR a la ruta exacta donde se descomprimió tu data
SOURCE_DIR = Path('/kaggle/working/dataset_2/Grazing_Running_Sitting Sheep_Classes')
TARGET_DIR = Path('/kaggle/working/dataset_plano_2')
CSV_PATH = Path('/kaggle/working/train2.csv')

# 2. Diccionario de codificación de clases
CLASS_MAP = {
    'Grazing': 0,
    'Running': 1,
    'Sitting': 2,
    'Standing': 3,
    'Walking': 4
}

# Crear la carpeta de destino si no existe
TARGET_DIR.mkdir(parents=True, exist_ok=True)

# Lista para almacenar los metadatos
metadata = []

# Extensiones de video válidas (puedes agregar más si es necesario)
video_extensions = {'.mp4', '.mov', '.avi', '.mkv'}

# 3. Recorrer la estructura de carpetas
for file_path in SOURCE_DIR.rglob('*'):
    if file_path.is_file() and file_path.suffix.lower() in video_extensions:
        
        # El nombre de la subcarpeta padre es la clase
        class_name = file_path.parent.name
        
        # Solo procesar si la carpeta coincide con una de tus clases objetivo
        if class_name in CLASS_MAP:
            # 4. Sanitizar el nombre del archivo
            # Elimina paréntesis y cambia espacios por guiones bajos
            clean_name = file_path.name.replace(' ', '_').replace('(', '').replace(')', '')
            
            # Para evitar que dos videos de distintas carpetas se llamen igual (ej. "001.mp4"),
            # le concatenamos la clase al inicio del nombre.
            final_name = f"{class_name.lower()}_{clean_name}"
            target_path = TARGET_DIR / final_name
            
            # 5. Mover el archivo (usamos move en lugar de copy para ahorrar los 20GB de Kaggle)
            shutil.move(str(file_path), str(target_path))
            
            # 6. Registrar en la lista para el CSV
            metadata.append({
                'filename': final_name,
                'label': CLASS_MAP[class_name]
            })

# 7. Crear y guardar el DataFrame
df = pd.DataFrame(metadata)
#df.to_csv(CSV_PATH, index=False)

print(f"✅ Procesamiento completado.")
print(f"📁 {len(df)} videos movidos a {TARGET_DIR}")
#print(f"📊 CSV generado en {CSV_PATH}")
display(df.head()) # Muestra las primeras filas para verificar

In [6]:
!zip -r ovejitas_dataset_final.zip dataset_plano_2/

ls: cannot access '/kaggle/tmp': No such file or directory


In [12]:
rm dataset_ovejitas.zip